# NB 64 — Concept source: custom core words vs FRH 8-language WordNet

Compares two ways of **constructing** the same four concepts (joy, sorrow,
safe, harm), then steers with each and compares the outcome, for both the
**multi-FRH** and **subspace** pipelines, at `λ ∈ {0, 0.4}`.

- **core** — the curated English word lists (Borda `core` field of
  `resources/64_custom_concepts.json`), tokenized directly (English only).
- **wn8lang** — the canonical FRH construction: a WordNet synset's lemmas in
  the 8-language Llama set (`SupportedLanguages.Llama`, added via
  `add_exomw()`), pooled into the concept. Synset map (canonical names):
  `joy→joy.n.01`, `sorrow→sorrow.n.01`, `safe→safe.a.01`,
  `harm→injury.n.01` (harm.n.01 aliases to injury.n.01: injury/hurt/harm/trauma).

The two sources differ in content: `core` is ~50 curated near-synonyms; a
WordNet synset is a handful of canonical lemmas plus their translations. So
this compares *concept source* — same concepts by name, different construction.

**Battery** (per source): 4 singles + 4 repels + 4 blends (from the same names).

**Metrics.** `directed effect` = `Σ sign·Δexpression` measured **self-
referentially** (each steered output against the concept it was steered with);
units differ by method/source, so read within a (method, source). The
cross-source-neutral axes are **lexical success** — presence of a *core* word
(the fixed semantic target for both sources) — and the **perplexity ratio**.

**Recipe:** bf16 Llama-3.1-8B, `k=4`, `steps=24`, `λ∈{0,0.4}`,
`max_token_count=4`, subspace `rank=8`, `MIN_LEMMAS_8=2` (wn8lang filter).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from subspace import ConceptSubspace, SubspaceUnembeddingRepresentation
from subspace.generation import (
    select_best_candidate,
    select_best_candidate_fluent,
    subspace_alignment,
)

In [ ]:
# -- parameters (papermill-overridable) -------------------------------------
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CONCEPTS_JSON = "resources/64_custom_concepts.json"
WORD_FIELD = "core"

MAX_TOKENS, K, STEPS = 4, 4, 24
RANK = 8
LAMBDAS = [0.0, 0.4]
MIN_LEMMAS_8 = 2  # wn8lang synsets are sparse; keep the filter permissive
SUFFIX = ""

SOURCES = ["core", "wn8lang"]
# synset values are the CANONICAL WordNet names (harm.n.01 aliases to injury.n.01)
SYNSET_MAP = {"joy": "joy.n.01", "sorrow": "sorrow.n.01",
              "safe": "safe.a.01", "harm": "injury.n.01"}

SINGLES = [("joy",), ("sorrow",), ("safe",), ("harm",)]
REPELS = [("safe", "harm"), ("harm", "safe"), ("joy", "sorrow"), ("sorrow", "joy")]
BLENDS = [("joy", "safe"), ("sorrow", "safe"), ("joy", "harm"), ("sorrow", "harm")]

SMOKE = True  # subset prompts + `_smoke` outputs; flip to False for the full run

In [ ]:
QUESTIONS = [
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Write a paragraph about the city.", "The city"),
    ("Continue this thought.", "Here is what happened next:"),
    ("Describe a person you met.", "The person I met was"),
]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


if SMOKE:
    QUESTIONS = QUESTIONS[:3]
    SUFFIX = SUFFIX or "_smoke"

PROMPTS = [chat(u, a) for u, a in QUESTIONS]
ENTRIES = {e["concept"]: e for e in json.loads(Path(CONCEPTS_JSON).read_text())}
WORDS = {name: ENTRIES[name][WORD_FIELD] for name in SYNSET_MAP}
print(f"{len(PROMPTS)} prompts | concepts: {list(WORDS)} | sources: {SOURCES}")

## Custom-concept builder (for the `core` source)

In [ ]:
SURFACE_VARIATIONS = ("{term}", " {term}", "{term} ", " {term} ")


def custom_concept_dataframe(tokenizer, name, words, max_token_count):
    """One-row synset-style DataFrame ({synset, tokens}) from a word list."""
    pad_id, bos_id = tokenizer.pad_token_id, tokenizer.bos_token_id
    rows = []
    for w in words:
        for v in SURFACE_VARIATIONS:
            ids = tokenizer(v.format(term=w.strip()), add_special_tokens=False)[
                "input_ids"
            ]
            if ids and ids[0] == bos_id:
                ids = ids[1:]
            if 0 < len(ids) <= max_token_count:
                rows.append(ids + [pad_id] * (max_token_count - len(ids)))
    if not rows:
        raise ValueError(f"no variation of {name!r} fits max_token_count={max_token_count}")
    return pd.DataFrame({"synset": [name], "tokens": [np.array(rows, dtype=np.int64)]})

## Load the model once, expose both representations

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16,
)
sur = SubspaceUnembeddingRepresentation(
    model=fur.model, unembedding=fur, data=fur.data, rank=RANK, method="pooled",
)
print("frame language set:", fur.data.language_codes.name)  # expect Llama (8 langs)

LOG_PATH = Path(f"resources/65_source_battery{SUFFIX}.jsonl")
LOG_PATH.write_text("")
harness = EvaluationHarness(fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS_8, max_token_count=MAX_TOKENS)
harness.member_lemmas("joy.n.01")  # force the WordNet parquet load out of the loops

## Build both concept sources (frame + subspace)

`core` from the curated word lists; `wn8lang` from the WordNet synset with
8-language lemmas (`get_concept_cached`). Concepts are keyed by `(source, name)`.

In [ ]:
fconcept, sconcept = {}, {}
core_wordset = {name: {w.strip().lower() for w in WORDS[name]} for name in WORDS}
rows = []
tok = fur.model.tokenizer
for name, synset in SYNSET_MAP.items():
    # core (custom words)
    df = custom_concept_dataframe(tok, name, WORDS[name], MAX_TOKENS)
    fconcept[("core", name)] = fur.compute_concept(df)
    sconcept[("core", name)] = sur.compute_concept(df)
    # wn8lang (WordNet synset, 8-language lemmas)
    fconcept[("wn8lang", name)] = fur.get_concept_cached(synset, MIN_LEMMAS_8, MAX_TOKENS)
    sconcept[("wn8lang", name)] = sur.get_concept_cached(synset, MIN_LEMMAS_8, MAX_TOKENS, method="pooled")
    for src in SOURCES:
        rows.append({"concept": name, "source": src,
                     "frame_k": fconcept[(src, name)].tensor.shape[-1],
                     "sub_rank": sconcept[(src, name)].rank})
print(pd.DataFrame(rows).to_string(index=False))

## Configs, guide builders (source-parametrized), and the subspace repel loop

In [ ]:
def make_configs():
    cfgs = []
    for (a,) in SINGLES:
        cfgs.append({"name": a, "case": "1-concept", "kind": "single", "parts": [(a, +1)]})
    for a, b in REPELS:
        cfgs.append({"name": f"{a}-{b}", "case": "2-concept", "kind": "repel", "parts": [(a, +1), (b, -1)]})
    for a, b in BLENDS:
        cfgs.append({"name": f"{a}+{b}", "case": "2-concept", "kind": "blend", "parts": [(a, +1), (b, +1)]})
    return cfgs


CONFIGS = make_configs()
print(f"{len(CONFIGS)} configs x {len(SOURCES)} sources x 2 methods x {len(LAMBDAS)} λ")

In [ ]:
@torch.inference_mode()
def subspace_repel_generate(rep, input_text, pos, neg, k, steps, fluency_weight=0.0):
    """Subspace top-k generation: attract `pos`, repel `neg` (‖B_pos·h‖−‖B_neg·h‖)."""
    input_text = [input_text] if isinstance(input_text, str) else input_text
    n = len(input_text)
    pos_basis = pos.basis.to(rep.model.device)
    neg_basis = neg.basis.to(rep.model.device)
    inputs = rep.model.make_input(input_text, padding=True)
    first, _, first_logprobs = rep._candidates(inputs["input_ids"], k)
    tokens = first.flatten(0, 1)
    cum_logprob = first_logprobs.flatten()
    probe = torch.zeros(n, 1)
    for _ in range(steps):
        sequences, last_hs, child_logprobs = rep._candidates(tokens, k)
        scores = subspace_alignment(last_hs, pos_basis) - subspace_alignment(last_hs, neg_basis)
        if fluency_weight:
            col_idx, probe = select_best_candidate_fluent(scores, cum_logprob, fluency_weight, n, k)
        else:
            col_idx, probe = select_best_candidate(scores, n, k)
        candidate_tokens = sequences.reshape(n, k, k, -1)
        tokens = candidate_tokens[np.arange(n), col_idx.cpu()].flatten(0, 1)
        if fluency_weight:
            r = torch.arange(n, device=cum_logprob.device) * k + col_idx.to(cum_logprob.device)
            cum_logprob = (cum_logprob[r].unsqueeze(-1) + child_logprobs[r]).flatten()
        if rep._is_generation_complete(tokens):
            break
    return rep.model.decode(tokens[::k]), probe.cpu()


def multifrh_steer(cfg, lam, source):
    guides = [fconcept[(source, name)] for name, _ in cfg["parts"]]
    weights = [float(sign) for _, sign in cfg["parts"]]
    normalize = None if len(guides) == 1 else "zscore"
    return fur.generate_with_topk_multi_guide(
        PROMPTS, guides=guides, weights=weights, k=K, steps=STEPS,
        normalize=normalize, fluency_weight=lam,
    )


def subspace_steer(cfg, lam, source):
    parts = cfg["parts"]
    if cfg["kind"] == "single":
        return sur.generate_with_topk_guide(
            PROMPTS, guide=sconcept[(source, parts[0][0])], k=K, steps=STEPS, fluency_weight=lam
        )
    if cfg["kind"] == "blend":
        guide = ConceptSubspace.blend([sconcept[(source, a)] for a, _ in parts])
        return sur.generate_with_topk_guide(
            PROMPTS, guide=guide, k=K, steps=STEPS, fluency_weight=lam
        )
    (a, _), (b, _) = parts
    return subspace_repel_generate(
        sur, PROMPTS, sconcept[(source, a)], sconcept[(source, b)], K, STEPS, fluency_weight=lam
    )

## Baseline + measurement helpers (expression self-referential; success = core lexicon)

In [ ]:
baseline_texts = fur.generate(PROMPTS, max_new_tokens=STEPS + 2, do_sample=False)
baseline_conts = [harness.continuation(t, p) for t, p in zip(baseline_texts, PROMPTS)]


def frame_expr(text, name, source):
    return harness.concept_expression(text, fconcept[(source, name)])


def subspace_expr(text, name, source):
    if not text.strip():
        return 0.0
    with torch.inference_mode():
        return float(sur.expression(text, sconcept[(source, name)]).float().mean())


EXPR = {"multifrh": frame_expr, "subspace": subspace_expr}
STEER = {"multifrh": multifrh_steer, "subspace": subspace_steer}


def ppl_ratio(cont, base_cont):
    if not cont.strip() or not base_cont.strip():
        return float("nan")
    return harness.perplexity(cont) / harness.perplexity(base_cont)


def log(record):
    with LOG_PATH.open("a") as fh:
        fh.write(json.dumps(record) + "\n")

In [ ]:
for source in SOURCES:
    for method in ("multifrh", "subspace"):
        expr = EXPR[method]
        for cfg in CONFIGS:
            attract = {w for name, sign in cfg["parts"] if sign > 0 for w in core_wordset[name]}
            for lam in LAMBDAS:
                texts, _ = STEER[method](cfg, lam, source)
                for i, (prompt, text) in enumerate(zip(PROMPTS, texts)):
                    cont = harness.continuation(text, prompt)
                    base = baseline_conts[i]
                    directed = 0.0
                    for name, sign in cfg["parts"]:
                        directed += sign * (expr(cont, name, source) - expr(base, name, source))
                    log({
                        "config": {"name": cfg["name"], "case": cfg["case"], "kind": cfg["kind"],
                                   "method": method, "source": source, "lam": lam},
                        "prompt": prompt, "continuation": cont,
                        "directed_effect": directed,
                        "success": harness.contains_any(cont, attract),
                        "ppl_ratio": ppl_ratio(cont, base),
                    })
                print(f"  {source:8s} {method:8s} {cfg['kind']:6s} {cfg['name']:14s} λ={lam} done")
print("battery complete ->", LOG_PATH)

## Results — core vs wn8lang, per method, per case, per λ

`success_rate` (presence of a core word) and `ppl_ratio` are the cross-source /
cross-method neutral axes. `directed_*` is self-referential (each source measured
against its own concept), so compare it within a (method, source).

In [ ]:
records = [json.loads(x) for x in LOG_PATH.read_text().strip().split("\n") if x]
rdf = pd.DataFrame(
    [{**r["config"], "directed": r["directed_effect"],
      "success": r["success"], "ppl_ratio": r["ppl_ratio"]} for r in records]
)

summary = (
    rdf.groupby(["method", "source", "case", "lam"])
    .agg(directed_total=("directed", "sum"), directed_mean=("directed", "mean"),
         success_rate=("success", "mean"), ppl_ratio=("ppl_ratio", "mean"),
         n=("directed", "size"))
    .reset_index()
)
summary.to_csv(f"resources/65_source_summary{SUFFIX}.csv", index=False)
print("=== core vs wn8lang, per method / case / λ ===")
print(summary.round(4).to_string(index=False))

print("\n=== source deltas (wn8lang − core) on the neutral axes, per method/case/λ ===")
piv = summary.pivot_table(index=["method", "case", "lam"], columns="source",
                          values=["success_rate", "ppl_ratio"])
for metric in ("success_rate", "ppl_ratio"):
    piv[(metric, "wn8lang−core")] = piv[(metric, "wn8lang")] - piv[(metric, "core")]
print(piv.round(4).to_string())

In [ ]:
CASES = ["1-concept", "2-concept"]
src_colors = {"core": "tab:green", "wn8lang": "tab:orange"}
panels = [("directed_mean", "directed effect (self-ref)\nper-method/source units", None),
          ("success_rate", "core-word lexical success", None),
          ("ppl_ratio", "perplexity ratio", 2.5)]
METHODS = ["multifrh", "subspace"]
PLOT_LAM = LAMBDAS[-1]  # operating point
x = np.arange(len(CASES))
width = 0.38
fig, axes = plt.subplots(len(METHODS), len(panels), figsize=(4 * len(panels), 3.4 * len(METHODS)))
for row, method in enumerate(METHODS):
    for col, (metric, label, guard) in enumerate(panels):
        ax = axes[row, col]
        for j, src in enumerate(SOURCES):
            vals = [summary[(summary.method == method) & (summary.source == src)
                            & (summary.case == c) & (summary.lam == PLOT_LAM)][metric].sum()
                    for c in CASES]
            ax.bar(x + (j - 0.5) * width, vals, width, label=src, color=src_colors[src])
        ax.set_xticks(x); ax.set_xticklabels(CASES)
        ax.set_title(f"{method} — {label}")
        ax.axhline(0, color="gray", lw=0.6)
        if guard:
            ax.axhline(guard, color="red", ls=":", lw=0.8)
        ax.legend()
plt.suptitle(f"Concept source: core vs wn8lang (λ={PLOT_LAM})")
plt.tight_layout()
plt.savefig(f"resources/65_source_battery{SUFFIX}.png", dpi=150)
plt.show()

## Notes

- **core** = curated English word lists (English-only tokens); **wn8lang** =
  WordNet synset lemmas across 8 languages (`SupportedLanguages.Llama`). Same
  concepts by name, different construction and content.
- **Neutral comparison axes:** `success_rate` (presence of a *core* word — the
  fixed semantic target for both sources) and `ppl_ratio` (fluency). The
  `directed_*` effect is self-referential (each source vs its own concept), so
  its magnitude is not comparable across sources; read it within a (method,
  source), or use the positive-direction sign.
- Synset map (canonical names): `joy.n.01`, `sorrow.n.01`, `safe.a.01`,
  `injury.n.01` (= harm.n.01 alias); `MIN_LEMMAS_8=2` keeps these sparse
  synsets in the WordNet filter.
- Flip `SMOKE = False` for the full 8-prompt battery.